In [74]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/across_cohort_descriptive_func.R')
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/ind_descriptive_func.R')

path.in <- '/home/jupyter/workspaces/Data/'

In [2]:
diabetes.activity.pid <- read.delim(
    paste0(path.in, 'Diabetes_activity_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
diabetes.sleep.pid <- read.delim(
    paste0(path.in, 'Diabetes_sleep_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
diabetes.hr.pid <- read.delim(
    paste0(path.in, 'Diabetes_hr_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );

insomnia.activity.pid <- read.delim(
    paste0(path.in, 'Insomnia_activity_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
insomnia.sleep.pid <- read.delim(
    paste0(path.in, 'Insomnia_sleep_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
insomnia.hr.pid <- read.delim(
    paste0(path.in, 'Insomnia_hr_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );

prostate.activity.pid <- read.delim(
    paste0(path.in, 'Prostate_activity_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
prostate.sleep.pid <- read.delim(
    paste0(path.in, 'Prostate_sleep_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
prostate.hr.pid <- read.delim(
    paste0(path.in, 'Prostate_hr_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );

healthy.activity.pid <- read.delim(
    paste0(path.in, 'Healthy_activity_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
healthy.sleep.pid <- read.delim(
    paste0(path.in, 'Healthy_sleep_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
healthy.hr.pid <- read.delim(
    paste0(path.in, 'Healthy_hr_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );

In [4]:
diabetes.pid <- Reduce(intersect, list(diabetes.activity.pid$x, diabetes.sleep.pid$x, diabetes.hr.pid$x))
length(diabetes.pid)
write.table(diabetes.pid, paste0(path.in, 'diabetes_final_pid.txt'), sep = '\t', quote = F, row.names = F)

insomnia.pid <- Reduce(intersect, list(insomnia.activity.pid$x, insomnia.sleep.pid$x, insomnia.hr.pid$x))
length(insomnia.pid)
write.table(insomnia.pid, paste0(path.in, 'insomnia_final_pid.txt'), sep = '\t', quote = F, row.names = F)

prostate.pid <- Reduce(intersect, list(prostate.activity.pid$x, prostate.sleep.pid$x, prostate.hr.pid$x))
length(prostate.pid)
write.table(prostate.pid, paste0(path.in, 'prostate_final_pid.txt'), sep = '\t', quote = F, row.names = F)

healthy.pid <- Reduce(intersect, list(healthy.activity.pid$x, healthy.sleep.pid$x, healthy.hr.pid$x))
length(healthy.pid)
write.table(healthy.pid, paste0(path.in, 'healthy_final_pid.txt'), sep = '\t', quote = F, row.names = F)


[1] 1657

[1] 1724

[1] 290

[1] 6912

In [9]:
# read data 
library(tidyverse)
library(bigrquery)

diabetes.pid.sql <- paste(diabetes.pid, collapse = ", ")
diabetes_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", diabetes.pid.sql, ")"
)
diabetes_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "diabetes_meta",
  "diabetes_meta_*.csv")
message(str_glue('The data will be written to {diabetes_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), diabetes_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  diabetes_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/diabetes_meta/diabetes_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [10]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/diabetes_meta/diabetes_meta_*.csv'
diabetes_meta <- read_bq_export_from_workspace_bucket(diabetes_meta_path)

dim(diabetes_meta)

head(diabetes_meta, 5) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/diabetes_meta/diabetes_meta_000000000000.csv.



[1] 1657   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
2356871,1985-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
1484073,1991-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
8882958,1949-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
8494108,1948-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
9956631,1964-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these


In [47]:
demographic_tbl_func(df = diabetes_meta)

# A tibble: 21 × 2
   Demographic                      Value        
   <chr>                            <chr>        
 1 N                                "1,657"      
 2 Age                              ""           
 3 18-44                            "170 (10%)"  
 4 45-64                            "629 (38%)"  
 5 >65                              "858 (52%)"  
 6 Ethnicity                        ""           
 7 Hispanic or Latino               "145 (9%)"   
 8 Not Hispanic or Latino           "1,465 (88%)"
 9 Others                           "47 (3%)"    
10 Race                             ""           
11 American Indian or Alaska Native "12 (1%)"    
12 Asian                            "48 (3%)"    
13 Black or African American        "177 (11%)"  
14 Middle Eastern or North African  "5 (<1%)"    
15 More than one population         "108 (7%)"   
16 Others                           "137 (8%)"   
17 White                            "1,170 (71%)"
18 Sex at birth                

NULL

In [43]:
table(diabetes_meta$ethnicity_concept_id)


  903079   903096  1586148 38003563 38003564 
       1       25       21      145     1465 

In [22]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

insomnia.pid.sql <- paste(insomnia.pid, collapse = ", ")
insomnia_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", insomnia.pid.sql, ")"
)
insomnia_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "insomnia_meta",
  "insomnia_meta_*.csv")
message(str_glue('The data will be written to {insomnia_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), insomnia_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  insomnia_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/insomnia_meta/insomnia_meta_*.csv. Use this path when reading the data into your notebooks in the future.

Auto-refreshing stale OAuth token.



In [23]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/insomnia_meta/insomnia_meta_*.csv'
insomnia_meta <- read_bq_export_from_workspace_bucket(insomnia_meta_path)

dim(insomnia_meta)

head(insomnia_meta, 5) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/insomnia_meta/insomnia_meta_000000000000.csv.



[1] 1724   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
3266772,1990-06-15 00:00:00 UTC,8527,White,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
9665628,1957-06-15 00:00:00 UTC,8527,White,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
4924465,1978-06-15 00:00:00 UTC,8516,Black or African American,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
1870586,1989-06-15 00:00:00 UTC,8657,American Indian or Alaska Native,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
1722029,1959-06-15 00:00:00 UTC,8527,White,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population


In [48]:
demographic_tbl_func(df = insomnia_meta)

# A tibble: 21 × 2
   Demographic                      Value        
   <chr>                            <chr>        
 1 N                                "1,724"      
 2 Age                              ""           
 3 18-44                            "294 (17%)"  
 4 45-64                            "588 (34%)"  
 5 >65                              "842 (49%)"  
 6 Ethnicity                        ""           
 7 Hispanic or Latino               "130 (8%)"   
 8 Not Hispanic or Latino           "1,563 (91%)"
 9 Others                           "31 (2%)"    
10 Race                             ""           
11 American Indian or Alaska Native "11 (1%)"    
12 Asian                            "28 (2%)"    
13 Black or African American        "112 (6%)"   
14 Middle Eastern or North African  "8 (<1%)"    
15 More than one population         "92 (5%)"    
16 Others                           "114 (7%)"   
17 White                            "1,359 (79%)"
18 Sex at birth                

NULL

In [29]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

pca.pid.sql <- paste(prostate.pid, collapse = ", ")
pca_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", pca.pid.sql, ")"
)
pca_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "pca_meta",
  "pca_meta_*.csv")
message(str_glue('The data will be written to {pca_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), pca_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  pca_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/pca_meta/pca_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [30]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/pca_meta/pca_meta_*.csv'

pca_meta <- read_bq_export_from_workspace_bucket(pca_meta_path)

dim(pca_meta)

head(pca_meta, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/pca_meta/pca_meta_000000000000.csv.



[1] 290  10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
3849859,1963-06-15 00:00:00 UTC,2e+09,More than one population,38003564,Not Hispanic or Latino,45880669,Male,2e+09,More than one population
3355707,1959-06-15 00:00:00 UTC,2e+09,More than one population,38003564,Not Hispanic or Latino,45880669,Male,2e+09,More than one population
5899152,1944-06-15 00:00:00 UTC,2e+09,More than one population,38003564,Not Hispanic or Latino,45880669,Male,2e+09,More than one population
6530673,1942-06-15 00:00:00 UTC,2e+09,More than one population,38003564,Not Hispanic or Latino,45880669,Male,2e+09,More than one population
6238215,1950-06-15 00:00:00 UTC,2e+09,More than one population,38003564,Not Hispanic or Latino,45880669,Male,2e+09,More than one population


In [49]:
demographic_tbl_func(df = pca_meta)

# A tibble: 18 × 2
   Demographic                      Value       
   <chr>                            <chr>       
 1 N                                "290"       
 2 Age                              ""          
 3 45-64                            "11 (4%)"   
 4 >65                              "279 (96%)" 
 5 Ethnicity                        ""          
 6 Hispanic or Latino               "10 (3%)"   
 7 Not Hispanic or Latino           "276 (95%)" 
 8 Others                           "4 (1%)"    
 9 Race                             ""          
10 American Indian or Alaska Native "2 (1%)"    
11 Asian                            "3 (1%)"    
12 Black or African American        "12 (4%)"   
13 Middle Eastern or North African  "1 (<1%)"   
14 More than one population         "6 (2%)"    
15 Others                           "11 (4%)"   
16 White                            "255 (88%)" 
17 Sex at birth                     ""          
18 Male                             "290 (100%)"


NULL

In [33]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

healthy.pid.sql <- paste(healthy.pid, collapse = ", ")
healthy_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", healthy.pid.sql, ")"
)
healthy_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "healthy_meta",
  "healthy_meta_*.csv")
message(str_glue('The data will be written to {healthy_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), healthy_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  healthy_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/healthy_meta/healthy_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [34]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
healthy_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/healthy_meta/healthy_meta_*.csv'

healthy_meta <- read_bq_export_from_workspace_bucket(healthy_meta_path)

dim(healthy_meta)

head(healthy_meta, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/healthy_meta/healthy_meta_000000000000.csv.



[1] 6912   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
7518347,1988-06-15 00:00:00 UTC,8515,Asian,38003564,Not Hispanic or Latino,45878463,Female,8515,Asian
2742566,1978-06-15 00:00:00 UTC,8515,Asian,38003564,Not Hispanic or Latino,45878463,Female,8515,Asian
1126203,1990-06-15 00:00:00 UTC,8515,Asian,38003564,Not Hispanic or Latino,45878463,Female,8515,Asian
3087887,1996-06-15 00:00:00 UTC,8515,Asian,38003564,Not Hispanic or Latino,45878463,Female,8515,Asian
1504870,2001-06-15 00:00:00 UTC,8515,Asian,38003564,Not Hispanic or Latino,45878463,Female,8515,Asian


In [50]:
demographic_tbl_func(df = healthy_meta)

# A tibble: 21 × 2
   Demographic                      Value        
   <chr>                            <chr>        
 1 N                                "6,912"      
 2 Age                              ""           
 3 18-44                            "1,926 (28%)"
 4 45-64                            "2,270 (33%)"
 5 >65                              "2,716 (39%)"
 6 Ethnicity                        ""           
 7 Hispanic or Latino               "833 (12%)"  
 8 Not Hispanic or Latino           "5,919 (86%)"
 9 Others                           "160 (2%)"   
10 Race                             ""           
11 American Indian or Alaska Native "48 (1%)"    
12 Asian                            "411 (6%)"   
13 Black or African American        "363 (5%)"   
14 Middle Eastern or North African  "34 (<1%)"   
15 More than one population         "510 (7%)"   
16 Others                           "677 (10%)"  
17 White                            "4,869 (70%)"
18 Sex at birth                

NULL

In [75]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

diabetes.pid.sql <- paste(diabetes.pid, collapse = ", ")

diabetes_survey_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (195771, 201254, 201820, 201826, 4008576, 4024659, 40482801, 40482883, 40484648, 40484649, 40485020, 4058243, 4063042, 4063043, 4099214, 4099334, 4099651, 4129519, 4130162, 4152858, 4193704, 4200875, 4202383, 4230254, 42537681, 4263902, 4274025, 4304377, 4311629, 43531007, 43531008, 43531009, 43531010, 436670, 443238, 443412, 44808385, 45757474, 45757789)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
                )
            AND 
                c_occurrence.person_id IN (", diabetes.pid.sql, ")
            ) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
diabetes_survey_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "diabetes_survey",
  "diabetes_survey_*.csv")
message(str_glue('The data will be written to {diabetes_survey_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), diabetes_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  diabetes_survey_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_survey/diabetes_survey_*.csv. Use this path when reading the data into your notebooks in the future.



In [76]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_survey_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/diabetes_survey/diabetes_survey_*.csv'

diabetes_survey <- read_bq_export_from_workspace_bucket(diabetes_survey_path)

dim(diabetes_survey)

head(diabetes_survey, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260504/diabetes_survey/diabetes_survey_000000000000.csv.



[1] 943454     20

person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,condition_start_datetime,condition_end_datetime,condition_type_concept_id,condition_type_concept_name,stop_reason,visit_occurrence_id,visit_occurrence_concept_name,condition_source_value,condition_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,condition_status_source_value,condition_status_concept_id,condition_status_concept_name
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
9706700,81902,Urinary tract infectious disease,68566005,SNOMED,2016-10-06 09:45:00 UTC,2016-10-06 09:45:00 UTC,38000280,Observation recorded from EHR,NA,5.4e+16,Outpatient Visit,N39.0,35209324,"Urinary tract infection, site not specified",N39.0,ICD10CM,NA,NA,NA
2171757,81902,Urinary tract infectious disease,68566005,SNOMED,2020-10-13 13:41:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7.0e+15,Outpatient Visit,N39.0,35209324,"Urinary tract infection, site not specified",N39.0,ICD10CM,NA,NA,NA
6256904,253009,Thoracogenic scoliosis,72992003,SNOMED,2022-07-18 11:38:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7.0e+15,Outpatient Visit,M41.34,45538756,"Thoracogenic scoliosis, thoracic region",M41.34,ICD10CM,NA,NA,NA
6256904,4275423,Supraventricular tachycardia,6456007,SNOMED,2021-03-17 00:00:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7.0e+15,Outpatient Visit,I47.1,35207781,Supraventricular tachycardia,I47.1,ICD10CM,NA,NA,NA
6256904,40479625,Atherosclerosis of artery,441574008,SNOMED,2021-09-15 10:48:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7.0e+15,Outpatient Visit,440.8,44831257,Atherosclerosis of other specified arteries,440.8,ICD9CM,NA,NA,NA


In [77]:
condition_func(df = diabetes_survey)

[1] 1657
[1] "non of the conditions"
[1] 87780
[1] "non of the conditions"
[1] 160
[1] 0.09656005
[1] "Hyperlipidemia"
[1] 26586
[1] 1115
[1] 0.6729028
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   1958922               432867 Hyperlipidemia       
2   2171757               432867 Hyperlipidemia       
3   1185387               432867 Hyperlipidemia       
[1] "obesity"
[1] 12780
[1] 824
[1] 0.4972842
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   3503647               433736 Obesity              
2   9706700               433736 Obesity              
3   6221727               433736 Obesity              
[1] "hypertension"
[1] 45343
[1] 1236
[1] 0.7459264
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name 
      <dbl>                <dbl> <chr>                 
1   6256904               320128 E

NULL

In [78]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

insomnia.pid.sql <- paste(insomnia.pid, collapse = ", ")

insomnia_survey_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (195771, 201254, 201820, 201826, 4008576, 4024659, 40482801, 40482883, 40484648, 40484649, 40485020, 4058243, 4063042, 4063043, 4099214, 4099334, 4099651, 4129519, 4130162, 4152858, 4193704, 4200875, 4202383, 4230254, 42537681, 4263902, 4274025, 4304377, 4311629, 43531007, 43531008, 43531009, 43531010, 436670, 443238, 443412, 44808385, 45757474, 45757789)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
                )
            AND 
                c_occurrence.person_id IN (", insomnia.pid.sql, ")
            ) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
insomnia_survey_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "insomnia_survey",
  "insomnia_survey_*.csv")
message(str_glue('The data will be written to {insomnia_survey_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), insomnia_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  insomnia_survey_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_survey/insomnia_survey_*.csv. Use this path when reading the data into your notebooks in the future.



In [79]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_survey_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_survey/insomnia_survey_*.csv'

insomnia_survey <- read_bq_export_from_workspace_bucket(insomnia_survey_path)

dim(insomnia_survey)

head(insomnia_survey, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_survey/insomnia_survey_000000000000.csv.



[1] 1088394      20

person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,condition_start_datetime,condition_end_datetime,condition_type_concept_id,condition_type_concept_name,stop_reason,visit_occurrence_id,visit_occurrence_concept_name,condition_source_value,condition_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,condition_status_source_value,condition_status_concept_id,condition_status_concept_name
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
6662338,433440,Dysthymia,78667006,SNOMED,2023-08-04 15:51:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7e+15,Outpatient Visit,F34.1,35207163,Dysthymic disorder,F34.1,ICD10CM,NA,NA,NA
4595896,440360,Pure hyperglyceridemia,267433009,SNOMED,2019-08-16 11:26:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7e+15,Outpatient Visit,272.1,44834565,Pure hyperglyceridemia,272.1,ICD9CM,NA,NA,NA
6256904,73754,Restless legs,32914008,SNOMED,2023-01-04 13:40:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7e+15,Outpatient Visit,G25.81,45586281,Restless legs syndrome,G25.81,ICD10CM,NA,NA,NA
2316011,4247084,Enteropathic arthritis,9350004,SNOMED,2023-01-09 10:23:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7e+15,Outpatient Visit,M07.60,45586846,"Enteropathic arthropathies, unspecified site",M07.60,ICD10CM,NA,NA,NA
6256904,435839,Lymphedema,234097001,SNOMED,2018-12-12 10:30:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7e+15,Outpatient Visit,I89.0,35207911,"Lymphedema, not elsewhere classified",I89.0,ICD10CM,NA,NA,NA


In [80]:
condition_func(df = insomnia_survey)

[1] 1724
[1] "non of the conditions"
[1] 71784
[1] "non of the conditions"
[1] 348
[1] 0.2018561
[1] "Hyperlipidemia"
[1] 22616
[1] 962
[1] 0.5580046
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   2316011               432867 Hyperlipidemia       
2   3259490               432867 Hyperlipidemia       
3   1958922               432867 Hyperlipidemia       
[1] "obesity"
[1] 11024
[1] 723
[1] 0.4193735
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   2256686               433736 Obesity              
2   3164897               433736 Obesity              
3   3266772               433736 Obesity              
[1] "hypertension"
[1] 36085
[1] 979
[1] 0.5678654
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name 
      <dbl>                <dbl> <chr>                 
1   1591345               320128 Esse

NULL

In [82]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

pca.pid.sql <- paste(prostate.pid, collapse = ", ")

pca_survey_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (195771, 201254, 201820, 201826, 4008576, 4024659, 40482801, 40482883, 40484648, 40484649, 40485020, 4058243, 4063042, 4063043, 4099214, 4099334, 4099651, 4129519, 4130162, 4152858, 4193704, 4200875, 4202383, 4230254, 42537681, 4263902, 4274025, 4304377, 4311629, 43531007, 43531008, 43531009, 43531010, 436670, 443238, 443412, 44808385, 45757474, 45757789)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
                )
            AND 
                c_occurrence.person_id IN (", pca.pid.sql, ")
            ) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
pca_survey_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "pca_survey",
  "pca_survey_*.csv")
message(str_glue('The data will be written to {pca_survey_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), pca_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  pca_survey_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_survey/pca_survey_*.csv. Use this path when reading the data into your notebooks in the future.



In [83]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_survey_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_survey/pca_survey_*.csv'

pca_survey <- read_bq_export_from_workspace_bucket(pca_survey_path)

dim(pca_survey)

head(pca_survey, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_survey/pca_survey_000000000000.csv.



[1] 141798     20

person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,condition_start_datetime,condition_end_datetime,condition_type_concept_id,condition_type_concept_name,stop_reason,visit_occurrence_id,visit_occurrence_concept_name,condition_source_value,condition_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,condition_status_source_value,condition_status_concept_id,condition_status_concept_name
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
6360268,4103295,Ventricular tachycardia,25569003,SNOMED,2021-09-10 00:00:00 UTC,NA,44786627,Primary Condition,NA,3.0e+16,Outpatient Visit,I47.2,35207782,Ventricular tachycardia,I47.2,ICD10CM,NA,NA,NA
1732187,40326053,Dysphonia,16617009,SNOMED,2015-06-17 00:00:00 UTC,NA,44786629,Secondary Condition,NA,7.0e+15,Office Visit,784.42,44832715,Dysphonia,784.42,ICD9CM,NA,NA,NA
1699984,4172970,Nontraumatic rotator cuff tear,424175006,SNOMED,2019-02-19 06:00:00 UTC,NA,32821,EHR billing record,NA,6.6e+16,Inpatient Visit,M75.122,45533934,"Complete rotator cuff tear or rupture of left shoulder, not specified as traumatic",M75.122,ICD10CM,NA,NA,NA
5533799,438383,Essential thrombocythemia,109994006,SNOMED,2014-11-24 16:01:00 UTC,NA,44786627,Primary Condition,NA,3.0e+16,Outpatient Visit,238.71,44833360,Essential thrombocythemia,238.71,ICD9CM,NA,NA,NA
9365803,317576,Coronary arteriosclerosis,53741008,SNOMED,2016-10-04 07:00:00 UTC,NA,0,No matching concept,NA,3.0e+16,Outpatient Visit,414.01,44830080,Coronary atherosclerosis of native coronary artery,414.01,ICD9CM,NA,NA,NA


In [84]:
condition_func(df = pca_survey)

[1] 290
[1] "non of the conditions"
[1] 14163
[1] "non of the conditions"
[1] 38
[1] 0.1310345
[1] "Hyperlipidemia"
[1] 4363
[1] 197
[1] 0.6793103
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   5224273               432867 Hyperlipidemia       
2   3029952               432867 Hyperlipidemia       
3   1569741               432867 Hyperlipidemia       
[1] "obesity"
[1] 878
[1] 75
[1] 0.2586207
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   2044151               433736 Obesity              
2   8702495               433736 Obesity              
3   4913734               433736 Obesity              
[1] "hypertension"
[1] 8280
[1] 206
[1] 0.7103448
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name 
      <dbl>                <dbl> <chr>                 
1   6360268               320128 Essential h

NULL

In [89]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

healthy.pid.sql <- paste(healthy.pid, collapse = ", ")
length(healthy.pid)

healthy_survey_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (195771, 201254, 201820, 201826, 4008576, 4024659, 40482801, 40482883, 40484648, 40484649, 40485020, 4058243, 4063042, 4063043, 4099214, 4099334, 4099651, 4129519, 4130162, 4152858, 4193704, 4200875, 4202383, 4230254, 42537681, 4263902, 4274025, 4304377, 4311629, 43531007, 43531008, 43531009, 43531010, 436670, 443238, 443412, 44808385, 45757474, 45757789)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
                )
            AND 
                c_occurrence.person_id IN (", healthy.pid.sql, ")
            ) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
healthy_survey_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "healthy_survey",
  "healthy_survey_*.csv")
message(str_glue('The data will be written to {healthy_survey_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), healthy_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  healthy_survey_path,
  destination_format = "CSV")

[1] 6912

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_survey/healthy_survey_*.csv. Use this path when reading the data into your notebooks in the future.



In [88]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
healthy_survey_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_survey/healthy_survey_*.csv'

healthy_survey <- read_bq_export_from_workspace_bucket(healthy_survey_path)

dim(healthy_survey)

head(healthy_survey, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_survey/healthy_survey_000000000000.csv.



[1]  0 20

person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,condition_start_datetime,condition_end_datetime,condition_type_concept_id,condition_type_concept_name,stop_reason,visit_occurrence_id,visit_occurrence_concept_name,condition_source_value,condition_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,condition_status_source_value,condition_status_concept_id,condition_status_concept_name
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>


In [87]:
condition_func(df = healthy_survey)

[1] 0
[1] "non of the conditions"
[1] 0
[1] "non of the conditions"
[1] 0
[1] NaN
[1] "Hyperlipidemia"
[1] 0
[1] 0
[1] NaN
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
  <chr>     <chr>                <chr>                
1 NA        NA                   NA                   
2 NA        NA                   NA                   
3 NA        NA                   NA                   
[1] "obesity"
[1] 0
[1] 0
[1] NaN
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
  <chr>     <chr>                <chr>                
1 NA        NA                   NA                   
2 NA        NA                   NA                   
3 NA        NA                   NA                   
[1] "hypertension"
[1] 0
[1] 0
[1] NaN
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
  <chr>     <chr>                <chr>                
1 NA        NA                   NA                   
2 NA        NA                   NA

NULL